# 当前论文的224×224四模型训练和MESSIDOR-2外部评估

当前论文的224×224四模型训练和MESSIDOR-2外部评估。原cell5为训练；6构建外部清单；7推断并保存结果。

来源为作者提供的APTOS_retest.ipynb，2026年10月15日按用途筛选。
代码与保存的输出逐格原样复制；不是重新运行的结果。原文件另行完整保留。
原来的训练开关、路径和依赖安装命令均保留，运行前需要配置对应数据与环境。
下面的编号是原文件从1开始的单元格编号。


## 原单元格 2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 原单元格 3


In [ ]:
import zipfile
import os

# 1. 定义压缩包在云盘里的位置
# 假设你直接传到了根目录，那就是这个路径：
zip_path = '/content/drive/MyDrive/split_data_preprocessed.zip'

# 2. 定义你想解压到哪里
# 建议解压到 Colab 的临时空间 (/content)，读取速度比直接读云盘快 10 倍！
extract_path = '/content'

# 3. 开始解压
if os.path.exists(extract_path):
  print("开始解压... (可能需要几分钟)")
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      zip_ref.extractall(extract_path)
  print("解压完成！")


## 原单元格 5


In [ ]:
# ============================================================
# Fixed-split APTOS training + evaluation for four DR models
# ============================================================

!pip -q install timm openpyxl

import os
import random
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    roc_auc_score
)

from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# 1. Basic config
# ============================================================

SEED = 42
NUM_CLASSES = 5
BATCH_SIZE = 32
EPOCHS = 40
LR = 3e-4
PATIENCE = 10
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 改成你的固定 split 文件夹
SPLIT_ROOT = Path("/content/split_data_preprocessed")
TRAIN_DIR = SPLIT_ROOT / "train"
VAL_DIR = SPLIT_ROOT / "val"

# 输出目录
OUT_DIR = Path("/content/drive/MyDrive/dr_fixed_split_results")
CKPT_DIR = OUT_DIR / "checkpoints"
PRED_DIR = OUT_DIR / "predictions"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, CKPT_DIR, PRED_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "EfficientNet-Lite0": {
        "timm_name": "efficientnet_lite0",
        "ckpt_path": CKPT_DIR / "best_fixedsplit_efficientnet_lite0.pth"
    },
    "MobileNetV3-Large": {
        "timm_name": "mobilenetv3_large_100",
        "ckpt_path": CKPT_DIR / "best_fixedsplit_mobilenetv3_large.pth"
    },
    "MobileViT-S": {
        "timm_name": "mobilevit_s",
        "ckpt_path": CKPT_DIR / "best_fixedsplit_mobilevit_s.pth"
    },
    "ResNet-50": {
        "timm_name": "resnet50",
        "ckpt_path": CKPT_DIR / "best_fixedsplit_resnet50.pth"
    }
}


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 尽量提高复现性
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# ============================================================
# 3. Dataset from fixed folders
# ============================================================

class FixedFolderDRDataset(Dataset):
    """
    读取固定文件夹:
    root/0/*.png
    root/1/*.png
    ...
    root/4/*.png

    返回:
    image tensor, label, image_path, image_id
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        class_dirs = [d for d in self.root_dir.iterdir() if d.is_dir()]
        class_dirs = sorted(class_dirs, key=lambda x: x.name)

        for class_dir in class_dirs:
            # 优化：提取文件夹开头的数字 (例如 '0_No_DR' -> 0)
            try:
                import re
                digit_match = re.match(r'^(\d+)', class_dir.name)
                if digit_match:
                    label = int(digit_match.group(1))
                else:
                    raise ValueError
            except:
                raise ValueError(f"Class folder must start with a numeric 0-4, got: {class_dir.name}")

            files = []
            for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.bmp"]:
                files.extend(class_dir.glob(ext))
                files.extend(class_dir.glob(ext.upper()))

            for img_path in sorted(files, key=lambda x: x.name):
                self.samples.append((str(img_path), label))

        if len(self.samples) == 0:
            raise ValueError(f"No images found in {self.root_dir}")

        self.df = pd.DataFrame(self.samples, columns=["path", "label"])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        image_id = Path(path).name
        return image, label, path, image_id


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


train_dataset = FixedFolderDRDataset(TRAIN_DIR, transform=train_transforms)
val_dataset = FixedFolderDRDataset(VAL_DIR, transform=val_transforms)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

print("\nTrain class counts:")
print(train_dataset.df["label"].value_counts().sort_index())

print("\nVal class counts:")
print(val_dataset.df["label"].value_counts().sort_index())

# 保存 split 记录，正式论文复现用
train_dataset.df.to_csv(TABLE_DIR / "fixed_train_split_manifest.csv", index=False)
val_dataset.df.to_csv(TABLE_DIR / "fixed_val_split_manifest.csv", index=False)


# Weighted sampler only for training
train_labels = train_dataset.df["label"].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# 4. Loss
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


# ============================================================
# 5. Metrics
# ============================================================

def compute_metrics(y_true, y_pred, probs=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    # per-class specificity
    per_class_rows = []
    report = classification_report(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0
    )

    for cls in range(NUM_CLASSES):
        TP = cm[cls, cls]
        FP = cm[:, cls].sum() - TP
        FN = cm[cls, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        per_class_rows.append({
            "class": cls,
            "precision": report[str(cls)]["precision"],
            "recall": report[str(cls)]["recall"],
            "f1": report[str(cls)]["f1-score"],
            "specificity": specificity,
            "support": report[str(cls)]["support"]
        })

    # Referable DR: 0-1 non-referable, 2-4 referable
    ref_true = np.isin(y_true, [2, 3, 4]).astype(int)
    ref_pred = np.isin(y_pred, [2, 3, 4]).astype(int)

    tn, fp, fn, tp = confusion_matrix(ref_true, ref_pred, labels=[0, 1]).ravel()

    ref_sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    ref_spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ref_prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    ref_f1 = 2 * ref_prec * ref_sens / (ref_prec + ref_sens) if (ref_prec + ref_sens) > 0 else np.nan
    ref_acc = (tp + tn) / (tp + tn + fp + fn)

    ref_auc = np.nan
    if probs is not None:
        ref_score = probs[:, 2] + probs[:, 3] + probs[:, 4]
        try:
            ref_auc = roc_auc_score(ref_true, ref_score)
        except:
            ref_auc = np.nan

    summary = {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "qwk": qwk,
        "referable_sensitivity": ref_sens,
        "referable_specificity": ref_spec,
        "referable_precision": ref_prec,
        "referable_f1": ref_f1,
        "referable_accuracy": ref_acc,
        "referable_auc": ref_auc,
        "referable_tp": tp,
        "referable_tn": tn,
        "referable_fp": fp,
        "referable_fn": fn
    }

    return summary, pd.DataFrame(per_class_rows), cm


# ============================================================
# 6. Evaluation and prediction CSV export
# ============================================================

def evaluate_and_save_predictions(model, loader, model_label, dataset_name, out_csv_path):
    model.eval()

    rows = []
    y_true_all = []
    y_pred_all = []
    probs_all = []

    with torch.no_grad():
        for images, labels, paths, image_ids in tqdm(loader, desc=f"Eval {model_label} on {dataset_name}"):
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)

            labels_np = labels.numpy()

            for i in range(len(labels_np)):
                row = {
                    "dataset": dataset_name,
                    "model": model_label,
                    "image_id": image_ids[i],
                    "image_path": paths[i],
                    "y_true": int(labels_np[i]),
                    "y_pred": int(preds[i]),
                    "prob_0": float(probs[i, 0]),
                    "prob_1": float(probs[i, 1]),
                    "prob_2": float(probs[i, 2]),
                    "prob_3": float(probs[i, 3]),
                    "prob_4": float(probs[i, 4]),
                }
                row["referable_true"] = int(row["y_true"] in [2, 3, 4])
                row["referable_pred"] = int(row["y_pred"] in [2, 3, 4])
                row["referable_score"] = row["prob_2"] + row["prob_3"] + row["prob_4"]
                rows.append(row)

            y_true_all.extend(labels_np.tolist())
            y_pred_all.extend(preds.tolist())
            probs_all.append(probs)

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(out_csv_path, index=False)

    probs_all = np.vstack(probs_all)
    summary, per_class_df, cm = compute_metrics(y_true_all, y_pred_all, probs_all)

    return pred_df, summary, per_class_df, cm


# ============================================================
# 7. Train one model
# ============================================================

def train_one_model(model_label, timm_name, ckpt_path):
    print("\n" + "=" * 100)
    print(f"Training {model_label} ({timm_name})")
    print("=" * 100)

    set_seed(SEED)

    model = timm.create_model(timm_name, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_qwk = -999
    best_epoch = -1
    patience_count = 0

    history = []

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for images, labels, _, _ in tqdm(train_loader, desc=f"{model_label} epoch {epoch+1}/{EPOCHS}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(loss.item())

        # validation
        _, val_summary, _, _ = evaluate_and_save_predictions(
            model,
            val_loader,
            model_label=model_label,
            dataset_name="APTOS_val",
            out_csv_path=OUT_DIR / "temp_val_predictions.csv"
        )

        current_qwk = val_summary["qwk"]
        current_acc = val_summary["accuracy"]
        current_macro_f1 = val_summary["macro_f1"]

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "val_accuracy": current_acc,
            "val_macro_f1": current_macro_f1,
            "val_qwk": current_qwk
        })

        print(
            f"Epoch {epoch+1}: "
            f"loss={np.mean(train_losses):.4f}, "
            f"val_acc={current_acc*100:.2f}%, "
            f"macro_f1={current_macro_f1*100:.2f}%, "
            f"qwk={current_qwk:.4f}"
        )

        if current_qwk > best_qwk:
            best_qwk = current_qwk
            best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_path)
            print(f"  Saved best model: QWK={best_qwk:.4f}")
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print("  Early stopping triggered.")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(TABLE_DIR / f"training_history_{model_label}.csv", index=False)

    print(f"Best epoch for {model_label}: {best_epoch}, best QWK={best_qwk:.4f}")
    return ckpt_path


# ============================================================
# 8. Main: train all four models on fixed split
# ============================================================

RUN_TRAINING = True  # 如果你只是用已有 checkpoint 重新评估，改成 False

if RUN_TRAINING:
    for model_label, cfg in MODEL_CONFIGS.items():
        train_one_model(
            model_label=model_label,
            timm_name=cfg["timm_name"],
            ckpt_path=cfg["ckpt_path"]
        )


# ============================================================
# 9. Final evaluation on fixed APTOS val split
# ============================================================

summary_rows = []
all_per_class = []
all_predictions = []

for model_label, cfg in MODEL_CONFIGS.items():
    print("\n" + "=" * 100)
    print(f"Final evaluation: {model_label}")
    print("=" * 100)

    model = timm.create_model(cfg["timm_name"], pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(cfg["ckpt_path"], map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    pred_csv = PRED_DIR / f"aptos_val_predictions_{model_label.replace('-', '_').replace(' ', '_').lower()}.csv"

    pred_df, summary, per_class_df, cm = evaluate_and_save_predictions(
        model,
        val_loader,
        model_label=model_label,
        dataset_name="APTOS_val",
        out_csv_path=pred_csv
    )

    summary_row = {"model": model_label, "dataset": "APTOS_val", "n": len(pred_df)}
    summary_row.update(summary)
    summary_rows.append(summary_row)

    per_class_df.insert(0, "model", model_label)
    per_class_df.insert(1, "dataset", "APTOS_val")
    all_per_class.append(per_class_df)

    cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in range(NUM_CLASSES)],
                         columns=[f"pred_{i}" for i in range(NUM_CLASSES)])
    cm_df.to_csv(TABLE_DIR / f"confusion_matrix_aptos_val_{model_label}.csv")

    all_predictions.append(pred_df)

summary_df = pd.DataFrame(summary_rows)
per_class_all_df = pd.concat(all_per_class, ignore_index=True)
pred_all_df = pd.concat(all_predictions, ignore_index=True)

summary_df.to_csv(TABLE_DIR / "aptos_val_summary_metrics.csv", index=False)
per_class_all_df.to_csv(TABLE_DIR / "aptos_val_per_class_metrics.csv", index=False)
pred_all_df.to_csv(PRED_DIR / "aptos_val_all_predictions_combined.csv", index=False)

print("\nAPTOS fixed-split summary:")
display(summary_df)

print("\nPer-class metrics:")
display(per_class_all_df)


## 原单元格 6


In [ ]:
# ============================================================
# Unzip and check MESSIDOR-2_Preprocessed for external validation
# Run this BEFORE the MESSIDOR-2 validation code
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import shutil
import pandas as pd

# -------------------------
# 1. Set zip path
# -------------------------
# 这里改成你的 zip 文件真实路径
# 常见情况是放在 MyDrive 下面
M2_ZIP_PATH = Path("/content/drive/MyDrive/MESSIDOR-2_Preprocessed.zip")

# 解压目标位置
EXTRACT_BASE = Path("/content/dataset")
EXTRACT_BASE.mkdir(parents=True, exist_ok=True)

EXPECTED_ROOT = EXTRACT_BASE / "MESSIDOR-2_Preprocessed"


# -------------------------
# 2. If path is wrong, search possible zip files
# -------------------------

if not M2_ZIP_PATH.exists():
    print("M2_ZIP_PATH not found:", M2_ZIP_PATH)
    print("\nSearching possible zip files under MyDrive...")

    candidates = list(Path("/content/drive/MyDrive").rglob("*MESSIDOR*Preprocessed*.zip"))
    candidates += list(Path("/content/drive/MyDrive").rglob("*messidor*preprocessed*.zip"))

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Cannot find MESSIDOR-2_Preprocessed.zip. "
            "Please check the zip filename and path."
        )

    print("\nFound possible zip files:")
    for i, p in enumerate(candidates):
        print(f"{i}: {p}")

    # 默认使用第一个候选
    M2_ZIP_PATH = candidates[0]
    print("\nUsing:", M2_ZIP_PATH)


# -------------------------
# 3. Unzip only if not already extracted
# -------------------------

def folder_has_images(root):
    root = Path(root)
    if not root.exists():
        return False
    image_exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]
    count = 0
    for ext in image_exts:
        count += len(list(root.rglob(ext)))
        count += len(list(root.rglob(ext.upper())))
    return count > 0


if EXPECTED_ROOT.exists() and folder_has_images(EXPECTED_ROOT):
    print("Already extracted:", EXPECTED_ROOT)
else:
    print("Unzipping:")
    print("From:", M2_ZIP_PATH)
    print("To:", EXTRACT_BASE)

    # 如果之前解压了一半，先删掉，避免混乱
    if EXPECTED_ROOT.exists():
        shutil.rmtree(EXPECTED_ROOT)

    with zipfile.ZipFile(M2_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_BASE)

    print("Unzip finished.")


# -------------------------
# 4. Detect correct M2_ROOT after unzip
# -------------------------

def has_class_folders(root):
    root = Path(root)
    if not root.exists():
        return False

    subfolders = [p for p in root.iterdir() if p.is_dir()]
    labels = set()

    for p in subfolders:
        first = p.name.split("_")[0]
        if first.isdigit():
            labels.add(int(first))

    return labels == {0, 1, 2, 3, 4}


possible_roots = []

# 正常情况：/content/dataset/MESSIDOR-2_Preprocessed/0...
possible_roots.append(EXPECTED_ROOT)

# 有些 zip 会多包一层：/content/dataset/MESSIDOR-2_Preprocessed/MESSIDOR-2_Preprocessed/0...
for p in EXTRACT_BASE.rglob("*"):
    if p.is_dir() and has_class_folders(p):
        possible_roots.append(p)

# 去重
possible_roots = list(dict.fromkeys(possible_roots))

valid_roots = [p for p in possible_roots if has_class_folders(p)]

if len(valid_roots) == 0:
    print("\nSubfolders under EXTRACT_BASE:")
    for p in sorted(EXTRACT_BASE.iterdir()):
        print(" -", p)

    raise ValueError(
        "Cannot find a valid MESSIDOR-2 root with class folders 0-4. "
        "Please check the zip structure."
    )

M2_ROOT = valid_roots[0]
print("\nDetected M2_ROOT:", M2_ROOT)


# -------------------------
# 5. Check folder structure and image count
# -------------------------

print("\nSubfolders under M2_ROOT:")
for p in sorted([x for x in M2_ROOT.iterdir() if x.is_dir()], key=lambda x: x.name):
    print(" -", p.name)

# 不应该是 train/val
bad_folders = {"train", "val", "test"}
folder_names_lower = {p.name.lower() for p in M2_ROOT.iterdir() if p.is_dir()}

if len(folder_names_lower.intersection(bad_folders)) > 0:
    raise ValueError(
        "M2_ROOT seems to contain train/val/test folders. "
        "External validation should use full MESSIDOR-2 class folders directly."
    )

image_exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]

rows = []
for class_dir in sorted([p for p in M2_ROOT.iterdir() if p.is_dir()], key=lambda x: x.name):
    label_str = class_dir.name.split("_")[0]

    if not label_str.isdigit():
        continue

    label = int(label_str)

    files = []
    for ext in image_exts:
        files.extend(class_dir.glob(ext))
        files.extend(class_dir.glob(ext.upper()))

    for f in sorted(files, key=lambda x: x.name):
        rows.append({
            "path": str(f),
            "label": label,
            "image_id": f.name
        })

manifest_df = pd.DataFrame(rows)

print("\nTotal images:", len(manifest_df))
print("\nClass distribution:")
print(manifest_df["label"].value_counts().sort_index())

display(manifest_df.head(10))

if len(manifest_df) != 1744:
    raise ValueError(
        f"MESSIDOR-2 image count is {len(manifest_df)}, not 1744. "
        "Please check whether the zip is complete or whether M2_ROOT is wrong."
    )

found_classes = set(manifest_df["label"].unique().tolist())
if found_classes != {0, 1, 2, 3, 4}:
    raise ValueError(f"Expected classes 0-4, but found {found_classes}")

print("\nAll checks passed.")
print("Use this path in external validation code:")
print("M2_ROOT =", M2_ROOT)

## 原单元格 7


In [ ]:
# ============================================================
# MESSIDOR-2 External Validation for 4 APTOS-trained Models
# Fixed APTOS checkpoints -> Full MESSIDOR-2 evaluation
# No training, no M2 split, no M2-trained checkpoint
# ============================================================

!pip -q install timm openpyxl

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    roc_auc_score
)

from google.colab import drive
drive.mount('/content/drive')


# -------------------------
# 1. Config
# -------------------------

SEED = 42
NUM_CLASSES = 5
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# 改成你的 MESSIDOR-2 预处理后路径
# 里面应该是 0 / 1 / 2 / 3 / 4 或者 0_No_DR / 1_Mild 这类文件夹
# TODO: Please update this path to the actual location of your MESSIDOR-2 preprocessed dataset.
M2_ROOT = Path("/content/dataset/MESSIDOR-2_Preprocessed")

# 输出路径
OUT_DIR = Path("/content/drive/MyDrive/dr_fixed_split_results/messidor2_external_validation")
PRED_DIR = OUT_DIR / "predictions"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, PRED_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# APTOS fixed-split checkpoints
MODEL_CONFIGS = [
    {
        "label": "EfficientNet-Lite0",
        "timm_name": "efficientnet_lite0",
        "ckpt_path": "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_efficientnet_lite0.pth"
    },
    {
        "label": "MobileNetV3-Large",
        "timm_name": "mobilenetv3_large_100",
        "ckpt_path": "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_mobilenetv3_large.pth"
    },
    {
        "label": "MobileViT-S",
        "timm_name": "mobilevit_s",
        "ckpt_path": "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_mobilevit_s.pth"
    },
    {
        "label": "ResNet-50",
        "timm_name": "resnet50",
        "ckpt_path": "/content/drive/MyDrive/dr_fixed_split_results/checkpoints/best_fixedsplit_resnet50.pth"
    }
]


# -------------------------
# 2. Seed
# -------------------------

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# -------------------------
# 3. Dataset
# -------------------------

class ExternalFolderDRDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.samples = []

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset root not found: {self.root_dir}")

        class_dirs = sorted(
            [d for d in self.root_dir.iterdir() if d.is_dir()],
            key=lambda x: x.name
        )

        if len(class_dirs) == 0:
            raise ValueError(f"No class folders found under: {self.root_dir}")

        for class_dir in class_dirs:
            # 支持文件夹名：0, 1, 2, 3, 4
            # 也支持：0_No_DR, 1_Mild, 2_Moderate 等
            label_str = class_dir.name.split("_")[0]

            if not label_str.isdigit():
                raise ValueError(
                    f"Class folder name must start with 0-4, got: {class_dir.name}"
                )

            label = int(label_str)

            if label < 0 or label >= NUM_CLASSES:
                raise ValueError(f"Invalid label {label} in folder: {class_dir.name}")

            files = []
            for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]:
                files.extend(class_dir.glob(ext))
                files.extend(class_dir.glob(ext.upper()))

            for p in sorted(files, key=lambda x: x.name):
                self.samples.append((str(p), label))

        if len(self.samples) == 0:
            raise ValueError(f"No image files found under: {self.root_dir}")

        self.df = pd.DataFrame(self.samples, columns=["path", "label"])

        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)

        image_id = Path(path).name

        return image, label, path, image_id


m2_dataset = ExternalFolderDRDataset(M2_ROOT)

print("\nMESSIDOR-2 dataset loaded.")
print("Total images:", len(m2_dataset))
print("Class distribution:")
print(m2_dataset.df["label"].value_counts().sort_index())

m2_dataset.df.to_csv(TABLE_DIR / "messidor2_manifest.csv", index=False)

if len(m2_dataset) != 1744:
    print("\nWarning: MESSIDOR-2 total image count is not 1744.")
    print("Please confirm whether your M2 folder is complete or filtered.")


m2_loader = DataLoader(
    m2_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# -------------------------
# 4. Metrics
# -------------------------

def compute_metrics(y_true, y_pred, probs=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0
    )

    per_class_rows = []

    for c in range(NUM_CLASSES):
        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP
        TN = cm.sum() - TP - FP - FN

        specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        per_class_rows.append({
            "class": c,
            "precision": report[str(c)]["precision"],
            "recall": report[str(c)]["recall"],
            "f1": report[str(c)]["f1-score"],
            "specificity": specificity,
            "support": report[str(c)]["support"]
        })

    # Referable DR: class 0-1 vs class 2-4
    ref_true = np.isin(y_true, [2, 3, 4]).astype(int)
    ref_pred = np.isin(y_pred, [2, 3, 4]).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        ref_true,
        ref_pred,
        labels=[0, 1]
    ).ravel()

    ref_sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    ref_spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ref_prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    ref_f1 = (
        2 * ref_prec * ref_sens / (ref_prec + ref_sens)
        if (ref_prec + ref_sens) > 0 else np.nan
    )
    ref_acc = (tp + tn) / (tp + tn + fp + fn)

    ref_auc = np.nan
    if probs is not None:
        ref_score = probs[:, 2] + probs[:, 3] + probs[:, 4]
        try:
            ref_auc = roc_auc_score(ref_true, ref_score)
        except Exception:
            ref_auc = np.nan

    summary = {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "qwk": qwk,
        "referable_sensitivity": ref_sens,
        "referable_specificity": ref_spec,
        "referable_precision": ref_prec,
        "referable_f1": ref_f1,
        "referable_accuracy": ref_acc,
        "referable_auc": ref_auc,
        "referable_tp": tp,
        "referable_tn": tn,
        "referable_fp": fp,
        "referable_fn": fn
    }

    per_class_df = pd.DataFrame(per_class_rows)

    return summary, per_class_df, cm


# -------------------------
# 5. Model loading
# -------------------------

def load_checkpoint_safely(model, ckpt_path):
    ckpt_path = Path(ckpt_path)

    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)

    # 兼容不同保存格式
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        elif "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt
    else:
        state_dict = ckpt

    # 兼容 DataParallel 的 module. 前缀
    new_state_dict = {}
    for k, v in state_dict.items():
        new_k = k.replace("module.", "")
        new_state_dict[new_k] = v

    model.load_state_dict(new_state_dict, strict=True)

    return model


# -------------------------
# 6. Evaluation
# -------------------------

def evaluate_external_model(model, loader, model_label):
    model.eval()

    rows = []
    y_true_all = []
    y_pred_all = []
    probs_all = []

    with torch.no_grad():
        for images, labels, paths, image_ids in tqdm(loader, desc=f"Evaluating {model_label} on MESSIDOR-2"):
            images = images.to(DEVICE)

            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)

            labels_np = labels.numpy()

            y_true_all.extend(labels_np.tolist())
            y_pred_all.extend(preds.tolist())
            probs_all.append(probs)

            for i in range(len(labels_np)):
                row = {
                    "dataset": "MESSIDOR-2",
                    "model": model_label,
                    "image_id": image_ids[i],
                    "image_path": paths[i],
                    "y_true": int(labels_np[i]),
                    "y_pred": int(preds[i]),
                    "prob_0": float(probs[i, 0]),
                    "prob_1": float(probs[i, 1]),
                    "prob_2": float(probs[i, 2]),
                    "prob_3": float(probs[i, 3]),
                    "prob_4": float(probs[i, 4]),
                }

                row["referable_true"] = int(row["y_true"] in [2, 3, 4])
                row["referable_pred"] = int(row["y_pred"] in [2, 3, 4])
                row["referable_score"] = row["prob_2"] + row["prob_3"] + row["prob_4"]

                rows.append(row)

    probs_all = np.vstack(probs_all)

    pred_df = pd.DataFrame(rows)
    summary, per_class_df, cm = compute_metrics(
        y_true_all,
        y_pred_all,
        probs_all
    )

    return pred_df, summary, per_class_df, cm


# -------------------------
# 7. Run all four models
# -------------------------

all_summary_rows = []
all_per_class_rows = []
all_prediction_dfs = []

for cfg in MODEL_CONFIGS:
    model_label = cfg["label"]
    timm_name = cfg["timm_name"]
    ckpt_path = cfg["ckpt_path"]

    print("\n" + "=" * 80)
    print(f"Model: {model_label}")
    print(f"timm name: {timm_name}")
    print(f"checkpoint: {ckpt_path}")
    print("=" * 80)

    model = timm.create_model(
        timm_name,
        pretrained=False,
        num_classes=NUM_CLASSES
    )

    model = load_checkpoint_safely(model, ckpt_path)
    model = model.to(DEVICE)

    pred_df, summary, per_class_df, cm = evaluate_external_model(
        model,
        m2_loader,
        model_label
    )

    # save predictions
    pred_csv = PRED_DIR / f"messidor2_predictions_{model_label.replace(' ', '_').replace('-', '_')}.csv"
    pred_df.to_csv(pred_csv, index=False)

    # save per-class metrics
    per_class_df.insert(0, "model", model_label)
    per_class_csv = TABLE_DIR / f"messidor2_per_class_{model_label.replace(' ', '_').replace('-', '_')}.csv"
    per_class_df.to_csv(per_class_csv, index=False)

    # save confusion matrix
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{i}" for i in range(NUM_CLASSES)],
        columns=[f"pred_{i}" for i in range(NUM_CLASSES)]
    )
    cm_csv = TABLE_DIR / f"messidor2_confusion_matrix_{model_label.replace(' ', '_').replace('-', '_')}.csv"
    cm_df.to_csv(cm_csv)

    # collect summary
    summary_row = {
        "dataset": "MESSIDOR-2",
        "model": model_label,
        "timm_name": timm_name,
        "checkpoint": ckpt_path,
        "n": len(pred_df)
    }
    summary_row.update(summary)
    all_summary_rows.append(summary_row)

    all_per_class_rows.append(per_class_df)
    all_prediction_dfs.append(pred_df)

    print("\nSummary:")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    print("\nPer-class metrics:")
    display(per_class_df)

    print("\nConfusion matrix:")
    display(cm_df)


# -------------------------
# 8. Save combined results
# -------------------------

summary_df = pd.DataFrame(all_summary_rows)
summary_path = TABLE_DIR / "messidor2_summary_metrics.csv"
summary_df.to_csv(summary_path, index=False)

combined_per_class_df = pd.concat(all_per_class_rows, ignore_index=True)
combined_per_class_path = TABLE_DIR / "messidor2_per_class_metrics_all_models.csv"
combined_per_class_df.to_csv(combined_per_class_path, index=False)

combined_predictions_df = pd.concat(all_prediction_dfs, ignore_index=True)
combined_predictions_path = PRED_DIR / "messidor2_all_predictions_combined.csv"
combined_predictions_df.to_csv(combined_predictions_path, index=False)

print("\n" + "=" * 80)
print("MESSIDOR-2 external validation finished.")
print("=" * 80)

print("\nSaved summary:")
print(summary_path)

print("\nSaved per-class metrics:")
print(combined_per_class_path)

print("\nSaved combined predictions:")
print(combined_predictions_path)

print("\nFinal summary:")
display(summary_df)
